In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20" bitsandbytes evaluate rouge_score gptqmodel optimum

# 1. DATA PIPELINE

In [ ]:
from datasets import load_dataset

raw = load_dataset("lighteval/med_paragraph_simplification")

print("splits + sizes:", {k: len(v) for k, v in raw.items()})
first = next(iter(raw.values()))
print("columns:", first.column_names)
print("\none example:")
print(first[0])

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("vocab size:", tok.vocab_size, "| pad:", tok.pad_token, "| eos:", tok.eos_token)

In [ ]:
PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.\n\n"
    "Medical text:\n{technical}\n\nPlain language:\n"
)

ex = raw["train"][0]
prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
full_text = prompt + ex["answer"] + tok.eos_token   # what the model will learn from: instruction+input, then the answer

enc = tok(full_text)

print("num tokens:", len(enc["input_ids"]))
print("first 20 ids:", enc["input_ids"][:20])
print("round-trip decode:", tok.decode(enc["input_ids"][:20]))

In [ ]:
import torch
from torch.utils.data import Dataset

MAX_LEN = 512

class MedSimplifyDataset(Dataset):
    def __init__(self, split):
        self.data = split          # a HF split, e.g. raw["train"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # TODO(you): same idea as Cell 4, but:
        #   1. ex = self.data[idx]; build prompt + full_text (as in Cell 4)
        #   2. tokenize WITH truncation=True, max_length=MAX_LEN   <- so the 878-token ones fit
        #   3. return a dict of 1-D tensors:
        #        {"input_ids": <tensor>, "attention_mask": <tensor>}
        #      The tokenizer gives python LISTS; wrap each with torch.tensor(..., dtype=torch.long)
        ex = self.data[idx]
        prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
        full_text = prompt + ex["answer"] + tok.eos_token
        enc = tok(full_text, truncation=True, max_length=MAX_LEN)
        return {
            "input_ids" : torch.tensor(enc["input_ids"]),
            "attention_mask" : torch.tensor(enc["attention_mask"])
        }

# quick self-test:
ds = MedSimplifyDataset(raw["train"])
print("dataset size:", len(ds))
item = ds[0]
print("keys:", item.keys())
print("input_ids:", item["input_ids"].shape, item["input_ids"].dtype)

In [ ]:
from torch.utils.data import DataLoader

ds = MedSimplifyDataset(raw["train"])
bad_loader = DataLoader(ds, batch_size=4)   # no collate_fn yet

try:
    batch = next(iter(bad_loader))
    print("input_ids:", batch["input_ids"].shape)
except Exception as e:
    print("ERROR:", type(e).__name__, "-", str(e)[:160])

In [ ]:
from dataclasses import dataclass

@dataclass
class PadCollator:
    pad_token_id: int

    def __call__(self, batch):
        # batch = list of dicts, each {"input_ids":[L_i], "attention_mask":[L_i]} with DIFFERENT L_i
        max_len = max(example["input_ids"].size(0) for example in batch)

        input_ids, masks = [], []
        for ex in batch:
            ids, mask = ex["input_ids"], ex["attention_mask"]
            pad_len = max_len - ids.size(0)          # how many pad slots this example needs

            # TODO: right-pad `ids` with self.pad_token_id, and `mask` with 0, each by pad_len.
            #   hint: torch.cat([ids,  torch.full((pad_len,), self.pad_token_id, dtype=torch.long)])
            #         torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])
            ids  = torch.cat([ids, torch.full((pad_len, ), self.pad_token_id, dtype=torch.long)])
            mask = torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])

            input_ids.append(ids)
            masks.append(mask)

        # TODO: stack the lists of 1-D tensors into 2-D [B, max_len] tensors (hint: torch.stack)
        return {"input_ids": torch.stack(input_ids), "attention_mask": torch.stack(masks)}


# test it with a real DataLoader:
collator = PadCollator(pad_token_id=tok.pad_token_id)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collator)
batch = next(iter(loader))
print("input_ids:", batch["input_ids"].shape)       # expect [4, max_len_in_batch]
print("attention_mask:", batch["attention_mask"].shape)
print("mask row 0 (1=real, 0=pad):", batch["attention_mask"][0].tolist()[-10:])

In [ ]:
def make_loaders(batch_size=4):
    collator = PadCollator(pad_token_id=tok.pad_token_id)
    train_dl = DataLoader(MedSimplifyDataset(raw["train"]),
                          batch_size=batch_size, shuffle=True,  collate_fn=collator)
    val_dl   = DataLoader(MedSimplifyDataset(raw["validation"]),
                          batch_size=batch_size, shuffle=False, collate_fn=collator)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()
print("train batches:", len(train_dl), "| val batches:", len(val_dl))
xb = next(iter(train_dl))
print("one batch:", xb["input_ids"].shape)

# 2. FINE TUNING

In [ ]:
# Load the model
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # half precision -> fits the T4, trains faster
    device_map="cuda",           # put it on the GPU
)
n_params = sum(p.numel() for p in model.parameters())
print(f"{model.config.model_type} loaded | {n_params/1e9:.2f}B params | device: {next(model.parameters()).device}")

In [ ]:
batch = next(iter(train_dl))
batch = {k: v.to("cuda") for k, v in batch.items()}   # move the batch onto the GPU (model is there)

# SANITY CHECK ONLY: wrap in no_grad so we do NOT build a full autograd graph over the
# whole base model. Without this, one forward at batch 4 x seq 512 allocates ~10GB of
# activations (every param is trainable here, before LoRA) and OOMs a 12GB card.
with torch.no_grad():
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["input_ids"]
    )

print("loss:", out.loss.item())
print("logits shape:", out.logits.shape)   # [batch, seq_len, vocab_size]

del out, batch
torch.cuda.empty_cache()   # free the sanity-check memory before training

In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install -q peft

In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=8,                  # adapter "rank" — bigger = more capacity + more params. 8 is a fine default.
    lora_alpha=16,        # a scaling factor on the adapters
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # inject adapters into the attention layers
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
from torch.amp import autocast, GradScaler

# memory savers (you OOM'd before — these prevent it):
model.gradient_checkpointing_enable()   # recompute instead of storing activations
model.enable_input_require_grads()
model.config.use_cache = False          # required alongside gradient checkpointing

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
scaler = GradScaler("cuda")             # scales fp16 gradients so they don't vanish to 0
model.train()                           # put model in training mode
print("ready to train")

In [ ]:
def make_labels(input_ids, attention_mask):
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100     # -100 tells the loss to IGNORE padding positions
    return labels

MAX_STEPS = 300   # ~1/3 epoch (3568 train examples / batch 4 = 892 steps/epoch); enough for a real loss drop
step = 0
for batch in train_dl:
    batch = {k: v.to("cuda") for k, v in batch.items()}
    labels = make_labels(batch["input_ids"], batch["attention_mask"])

    optimizer.zero_grad()
    with autocast("cuda", dtype=torch.float16):
      out = model(
          input_ids=batch["input_ids"],
          attention_mask=batch["attention_mask"],
          labels=labels
      )
      loss = out.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    step += 1
    if step % 10 == 0:
        print(f"step {step:3d} | loss {loss.item():.3f}")
    if step >= MAX_STEPS:
        break
print("training done")

In [ ]:
model.eval()                              # eval mode (turns off dropout etc.)
model.gradient_checkpointing_disable()
model.config.use_cache = True             # re-enable the KV cache for fast generation — Day 3 is ALL about this

val_ex = raw["validation"][0]
prompt = PROMPT_TEMPLATE.format(technical=val_ex["query"])   # note: prompt only, NO answer this time
inputs = tok(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():                     # no training here -> don't track gradients (faster, less memory)
    gen = model.generate(**inputs, max_new_tokens=200, do_sample=False)

new_tokens = gen[0][inputs["input_ids"].shape[1]:]   # slice off the prompt; keep only what the model generated
print("=== MODEL OUTPUT ===")
print(tok.decode(new_tokens, skip_special_tokens=True))
print("\n=== HUMAN REFERENCE ===")
print(val_ex["answer"][:500])

In [ ]:
import os
model.save_pretrained("lucidrt-lora")
tok.save_pretrained("lucidrt-lora")
print("saved:", os.listdir("lucidrt-lora"))
size_mb = sum(os.path.getsize(os.path.join("lucidrt-lora", f)) for f in os.listdir("lucidrt-lora")) / 1e6
print(f"adapter size: {size_mb:.1f} MB")   # tiny! just the adapters, not the 1.5B base

# 3. KV Caching

In [ ]:
import time

# naive generation
def generate_naive(prompt, max_new_tokens=50):
    model.eval()
    input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(input_ids=input_ids, use_cache=False)   # recompute the WHOLE sequence every step
        logits = out.logits
        next_logits = logits[:, -1, :]
        next_id = next_logits.argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)

    return input_ids

prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

t0 = time.time()
out_ids = generate_naive(prompt, max_new_tokens=50)
dt = time.time() - t0
print(f"50 tokens in {dt:.2f}s  ->  {50/dt:.1f} tokens/sec")
print("---")
print(tok.decode(out_ids[0], skip_special_tokens=True)[-300:])

In [ ]:
def generate_cached(prompt, max_new_tokens=50):
    model.eval()
    input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")

    generated = input_ids
    cur = input_ids        # first pass: feed the whole prompt
    past = None            # the KV cache; empty to start

    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(input_ids=cur, past_key_values=past, use_cache=True)

        past = out.past_key_values
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        cur = next_id  # next pass feeds only the new token, the cache supplies all the earlier ones

    return generated

In [ ]:
# benchmark: naive vs cached, same prompt, same output length
prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

t0 = time.time(); naive_ids  = generate_naive(prompt, 50);  t_naive  = time.time() - t0
t0 = time.time(); cached_ids = generate_cached(prompt, 50); t_cached = time.time() - t0

print(f"naive :  {t_naive:.2f}s  ->  {50/t_naive:.1f} tok/s")
print(f"cached:  {t_cached:.2f}s  ->  {50/t_cached:.1f} tok/s")
print(f"speedup: {t_naive/t_cached:.2f}x")
print("same output? ", torch.equal(naive_ids, cached_ids))   # should be True — same tokens, just faster

# 4. Quantization

In [ ]:
from transformers import BitsAndBytesConfig

model_int8 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    # note: NO .to("cuda") — bitsandbytes places it on the GPU for you
)
model_int8.eval()

print(f"fp16 footprint: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"int8 footprint: {model_int8.get_memory_footprint()/1e9:.2f} GB")

In [ ]:
def generate_cached(prompt, max_new_tokens=50, gen_model=None):
  gen_model = gen_model or model          # <- default to the fp16 model
  model.eval()
  input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")

  generated = input_ids
  cur = input_ids        # first pass: feed the whole prompt
  past = None            # the KV cache; empty to start

  for _ in range(max_new_tokens):
      with torch.no_grad():
          out = gen_model(input_ids=cur, past_key_values=past, use_cache=True)

      past = out.past_key_values
      next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
      generated = torch.cat([generated, next_id], dim=1)
      cur = next_id  # next pass feeds only the new token, the cache supplies all the earlier ones

  return generated

In [ ]:
  prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

  t0 = time.time(); fp16_ids = generate_cached(prompt, 50);                   t_fp16 = time.time() - t0
  t0 = time.time(); int8_ids = generate_cached(prompt, 50, gen_model=model_int8); t_int8 = time.time() - t0

  print(f"fp16 cached:  {t_fp16:.2f}s  ->  {50/t_fp16:.1f} tok/s")
  print(f"int8 cached:  {t_int8:.2f}s  ->  {50/t_int8:.1f} tok/s")
  print("outputs identical? ", torch.equal(fp16_ids, int8_ids))
  print("--- int8 output ---")
  print(tok.decode(int8_ids[0], skip_special_tokens=True)[-200:])

In [ ]:
import time, torch

def profile_split(prompt, max_new_tokens=50):
    model.eval()
    input_ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    n_prompt = input_ids.shape[1]

    # warm-up: first CUDA launch is slow; run once and throw it away
    with torch.no_grad():
        _ = model(input_ids=input_ids, use_cache=True)
    torch.cuda.synchronize()

    # ---- PREFILL: one pass over the WHOLE prompt ----
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        out = model(input_ids=input_ids, use_cache=True)
    torch.cuda.synchronize(); t_prefill = time.time() - t0

    past = out.past_key_values
    cur  = out.logits[:, -1, :].argmax(-1, keepdim=True)

    # ---- DECODE: one token at a time, from the cache ----
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            out = model(input_ids=cur, past_key_values=past, use_cache=True)
        past = out.past_key_values
        cur  = out.logits[:, -1, :].argmax(-1, keepdim=True)
    torch.cuda.synchronize(); t_decode = time.time() - t0

    total = t_prefill + t_decode
    print(f"prompt length:        {n_prompt} tokens")
    print(f"prefill:            {t_prefill*1000:8.1f} ms  ({t_prefill/total*100:4.1f}%)  for all {n_prompt} tokens")
    print(f"decode ({max_new_tokens} tokens):   {t_decode*1000:8.1f} ms  ({t_decode/total*100:4.1f}%)")
    print(f"  per prompt token (prefill): {t_prefill/n_prompt*1000:6.3f} ms")
    print(f"  per output token (decode):  {t_decode/max_new_tokens*1000:6.3f} ms")

prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])
profile_split(prompt, max_new_tokens=50)

In [ ]:
profile_split(prompt, max_new_tokens=200)

In [ ]:
def batched_throughput(prompt, batch_size, max_new_tokens=50):
    model.eval()
    ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
    input_ids = ids.repeat(batch_size, 1)        # [B, L] — B identical prompts

    try:
        torch.cuda.synchronize(); t0 = time.time()
        cur, past = input_ids, None
        for _ in range(max_new_tokens):
            with torch.no_grad():
                out = model(input_ids=cur, past_key_values=past, use_cache=True)
            past = out.past_key_values
            cur  = out.logits[:, -1, :].argmax(-1, keepdim=True)   # [B, 1]
        torch.cuda.synchronize(); dt = time.time() - t0

        total = batch_size * max_new_tokens
        print(f"batch={batch_size:2d} | {dt:5.2f}s | {total/dt:7.1f} tok/s TOTAL | {max_new_tokens/dt:5.1f} tok/s per-seq")
    except torch.cuda.OutOfMemoryError:
        print(f"batch={batch_size:2d} | OOM — hit the memory ceiling")
        torch.cuda.empty_cache()

prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])
for b in [1, 2, 4, 8, 16]:
    batched_throughput(prompt, b, max_new_tokens=50)

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

In [ ]:
def eval_quality(gen_model, n=40, max_new_tokens=64, batch_size=8):
    tok.padding_side = "left"   # decoder-only models MUST left-pad for batched generation,
                                # so every sequence's newest token sits at the right edge
    data = raw["validation"].select(range(n))
    preds, refs = [], []

    for i in range(0, n, batch_size):
        chunk   = data.select(range(i, min(i + batch_size, n)))
        prompts = [PROMPT_TEMPLATE.format(technical=ex["query"]) for ex in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True,
                  truncation=True, max_length=512).to("cuda")

        with torch.no_grad():
            out = gen_model.generate(**enc, max_new_tokens=max_new_tokens,
                                      do_sample=False,              # greedy = matches our argmax loop
                                      pad_token_id=tok.pad_token_id)

        gen_only = out[:, enc["input_ids"].shape[1]:]              # strip the prompt, keep new tokens
        preds.extend(tok.batch_decode(gen_only, skip_special_tokens=True))
        refs.extend([ex["answer"] for ex in chunk])

    scores = rouge.compute(predictions=preds, references=refs)
    return {k: round(v, 4) for k, v in scores.items()}

# print("fp16:", eval_quality(model,      n=40))
# print("int8:", eval_quality(model_int8, n=40))

In [ ]:
!pip install -q autoawq

In [ ]:
model_awq = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    device_map="cuda",
)

In [ ]:
prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])

for name, m in [("fp16", model), ("int8", model_int8), ("int4-awq", model_awq)]:
  torch.cuda.synchronize(); t0 = time.time()
  _ = generate_cached(prompt, 50, gen_model=m)
  torch.cuda.synchronize(); dt = time.time() - t0
  print(f"{name:9s}: {dt:5.2f}s  ->  {50/dt:5.1f} tok/s")

In [ ]:
print("int4-awq:", eval_quality(model_awq, n=40))

In [ ]:
print(f"fp16 footprint: {model.get_memory_footprint()/1e9:.2f} GB")
print(f"int4 footprint: {model_awq.get_memory_footprint()/1e9:.2f} GB")

In [ ]:
prompt = PROMPT_TEMPLATE.format(technical=raw["validation"][0]["query"])
for name, m in [("fp16", model), ("int8", model_int8), ("int4-awq", model_awq)]:
  torch.cuda.synchronize(); t0 = time.time()
  _ = generate_cached(prompt, 50, gen_model=m)
  torch.cuda.synchronize(); dt = time.time() - t0
  print(f"{name:9s}: {dt:5.2f}s  ->  {50/dt:5.1f} tok/s")

# 5. Apples-to-apples quality (separating fine-tuning from quantization)

Latency & memory in sections 3-4 are not confounded - quantizing changes speed/size the same way whether or not the model is fine-tuned. Only **quality** was mixed (fp16 = fine-tuned; int8/int4 = base).

Here each number changes exactly **one** variable, and models are loaded **one at a time and freed** before the next, so this fits a 12 GB card:
- **Quantization effect** - base fp16 -> int8 -> int4.
- **Fine-tuning effect** - base fp16 -> fine-tuned fp16.

int8 is slow (and may be unstable on a TITAN V / sm_70); int8 and int4 are wrapped so a failure won't stop the rest.

In [ ]:
import gc
from transformers import BitsAndBytesConfig
N = 25   # eval sample size; keep the SAME for all variants (raise for a tighter estimate)
qual = {}

def _free(*names):
    for n in names:
        if n in globals(): del globals()[n]
    gc.collect(); torch.cuda.empty_cache()

# 5a. fine-tuned fp16 -- the LoRA `model` is still resident from training
model.config.use_cache = True
qual['ft_fp16'] = eval_quality(model, n=N)['rougeL']
print('ft    fp16    :', qual['ft_fp16'])

# free EVERYTHING on the GPU so we can reload base variants one at a time (12GB-safe)
_free('model', 'model_int8', 'model_awq')

In [ ]:
# 5b. base fp16 (NOT fine-tuned)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map='cuda').eval()
qual['base_fp16'] = eval_quality(base, n=N)['rougeL']
print('base  fp16    :', qual['base_fp16'])
_free('base')

In [ ]:
# 5c. base int8 (reload fresh) -- SLOW; may be unstable on sm_70, so it's wrapped
try:
    m8 = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=BitsAndBytesConfig(load_in_8bit=True)).eval()
    qual['base_int8'] = eval_quality(m8, n=N)['rougeL']
    _free('m8')
except Exception as e:
    print('int8 eval skipped:', type(e).__name__, str(e)[:80]); qual['base_int8'] = None
print('base  int8    :', qual['base_int8'])

In [ ]:
# 5d. base int4-AWQ (reload fresh). Needs the AWQ backend; if missing/unsupported
# (AWQ kernels may not cover sm_70), we fall back to the known value from an earlier run.
try:
    m4 = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct-AWQ', device_map='cuda').eval()
    qual['base_int4'] = eval_quality(m4, n=N)['rougeL']
    _free('m4')
except Exception as e:
    print('int4-awq eval skipped:', type(e).__name__); qual['base_int4'] = 0.0969
print('base  int4-awq:', qual['base_int4'])

In [ ]:
# 5e. Summary -- each comparison changes only ONE variable
print('=== QUALITY (ROUGE-L, n=%d, greedy) ===' % N)
for k in ['base_fp16','base_int8','base_int4','ft_fp16']:
    print('  %-10s: %s' % (k, qual.get(k)))
print()
print('QUANTIZATION effect (base): fp16 %s -> int8 %s -> int4 %s'
      % (qual.get('base_fp16'), qual.get('base_int8'), qual.get('base_int4')))
print('FINE-TUNING effect (fp16):  base %s -> fine-tuned %s'
      % (qual.get('base_fp16'), qual.get('ft_fp16')))